# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth refreshing when it is still **visible** in search
(it earns impressions) **and** shows at least one fixable **problem** — it's **stale** (not
updated in a long time), it **under-earns clicks** for its ranking position (low CTR), it's
**thin** (short but still getting impressions), or it's an **ageing page-one page** whose ranking
can slip. Rank pages by *how many problems* they have, weighted by *how visible* they are: a
broken page nobody sees is not urgent.

**Reason codes** (each fires from one readable condition, using only signals known **before** any
refresh decision):

| Reason code | Fires when |
|---|---|
| `stale_visible_page` | `days_since_last_update >= 180` **and** `impressions_90d >= 500` |
| `low_ctr_visible_page` | `impressions_90d >= 500`, `0 < avg_position <= 20`, `ctr < 0.5` |
| `thin_visible_page` | `0 < word_count < 1200` **and** `impressions_90d >= 250` |
| `page_one_decay_risk` | `0 < avg_position <= 10` **and** `content_age_days >= 180` |
| `general_refresh_review` | none of the above fired (score 0 — bottom of the queue) |

**Kept OUT of the score (leakage guard):** `trend_direction`, `trend_pct`, and the
`*_last_30d` / `*_prev_30d` windows (they *define* or overlap the label), plus the product fields
`provider_used` / `model_used`. The label `is_declining_label = (trend_direction == "down")` is
computed for **evaluation only**.

**Gotchas respected (from `docs/data-dictionary.md`):** `avg_position == 0` means *no data*, so
it's guarded out with `> 0`; `ctr` is a ×100 percentage, so `ctr < 0.5` means "under 0.5%";
missing `word_count` is treated as unknown (`0`) so it can't fire `thin_visible_page`.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Find the repo root so paths work whether this runs from work/notebooks/ or the repo root.
root = Path.cwd()
for _ in range(6):
    if (root / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        break
    root = root.parent
RAW = root / "data" / "raw" / "content_refresh_anonymized.csv"
assert RAW.exists(), "Starter CSV not found — open this notebook from inside the repo."

df = pd.read_csv(RAW)

# Label — EVALUATION ONLY. Never a score input (it is derived from trend_direction).
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# The handful of signals the rule reads, cleaned per the data-dictionary gotchas.
imp = df["impressions_90d"].fillna(0)
upd = df["days_since_last_update"].fillna(0)
pos = df["avg_position"].fillna(0)     # 0 == "no position data"
ctr = df["ctr"].fillna(0)              # ×100 percentage: 0.5 == 0.5%
wc  = df["word_count"].fillna(0)       # 0 == unknown length
age = df["content_age_days"].fillna(0)

# Reason-code flags — one plain condition each.
df["stale_visible_page"]   = ((upd >= 180) & (imp >= 500)).astype(int)
df["low_ctr_visible_page"] = ((imp >= 500) & (pos > 0) & (pos <= 20) & (ctr < 0.5)).astype(int)
df["thin_visible_page"]    = ((wc > 0) & (wc < 1200) & (imp >= 250)).astype(int)
df["page_one_decay_risk"]  = ((pos > 0) & (pos <= 10) & (age >= 180)).astype(int)

FLAGS = ["stale_visible_page", "low_ctr_visible_page", "thin_visible_page", "page_one_decay_risk"]

def reason_codes(row):
    fired = [f for f in FLAGS if row[f]]
    return "|".join(fired) if fired else "general_refresh_review"

def suggested_action(codes):
    s = set(codes.split("|"))
    if "thin_visible_page" in s:                          return "expand_and_refresh"
    if "low_ctr_visible_page" in s:                       return "refresh_and_review_ctr"
    if s & {"stale_visible_page", "page_one_decay_risk"}: return "refresh"
    return "monitor"

print(f"{len(df):,} pages loaded | declining base rate: {df['is_declining_label'].mean():.3f}")
print("flag fire counts:", {f: int(df[f].sum()) for f in FLAGS})

30,000 pages loaded | declining base rate: 0.542
flag fire counts: {'stale_visible_page': 17, 'low_ctr_visible_page': 9759, 'thin_visible_page': 82, 'page_one_decay_risk': 7076}


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Transparent score: rank by how many problems a page has (hand-picked integer weights,
# NOT fitted), then break ties by visibility. Bounded on purpose so a few giant pages can't
# swamp the queue — you can read the formula and predict its ranking.
weights = {"stale_visible_page": 2, "low_ctr_visible_page": 2,
           "thin_visible_page": 1, "page_one_decay_risk": 1}
df["problem_score"]   = sum(w * df[f] for f, w in weights.items())          # 0..6
df["visibility_pct"]  = df["impressions_90d"].fillna(0).rank(pct=True)      # 0..1 tie-breaker
# problems lead; visibility only orders pages that tie on problems; no problems -> bottom.
df["baseline_action_score"] = np.where(df["problem_score"] > 0,
                                        df["problem_score"] + df["visibility_pct"], 0.0)

df["reason_codes"]     = df.apply(reason_codes, axis=1)
df["suggested_action"] = df["reason_codes"].map(suggested_action)
df["baseline_rank"]    = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

cols = ["content_id", "client_id", "baseline_rank", "baseline_action_score", "problem_score",
        "visibility_pct", *FLAGS, "reason_codes", "suggested_action", "is_declining_label",
        "impressions_90d", "clicks_90d", "avg_position", "ctr",
        "days_since_last_update", "content_age_days", "word_count", "trend_direction"]
queue = df[cols].sort_values("baseline_rank").reset_index(drop=True)

# Regenerate the CSV on every run. work/outputs/*.csv is gitignored, so data never enters git
# (the CI leak-guard also blocks dataset CSVs) — this file is a build artifact, not a commit.
OUT = root / "work" / "outputs" / "baseline_action_score.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(OUT, index=False)
print(f"Wrote {len(queue):,} ranked rows -> {OUT.relative_to(root)}")
print(f"pages with >=1 problem (scored above 0): {(df['problem_score'] > 0).sum():,}")

# Honest metric: precision@K against the base-rate floor.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

y = df["is_declining_label"].values
base = y.mean()
for k in (20, 50, 100):
    p = precision_at_k(df["baseline_action_score"], y, k)
    print(f"Precision@{k}: {p:.3f}   (base rate {base:.3f}, lift x{p/base:.2f})")

Wrote 30,000 ranked rows -> work/outputs/baseline_action_score.csv
pages with >=1 problem (scored above 0): 13,508
Precision@20: 0.900   (base rate 0.542, lift x1.66)
Precision@50: 0.540   (base rate 0.542, lift x1.00)
Precision@100: 0.470   (base rate 0.542, lift x0.87)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Takeaway:** the top of the queue is dominated by *stale + still-visible* and *low-CTR page-one*
pages — all `high` confidence because two or more reason codes agree. The `would_be_wrong_if`
column is the honest hedge printed next to every pick, so the list stays trustworthy to a human.

In [3]:
top20 = queue.head(20).copy()
p99_imp = df["impressions_90d"].quantile(0.99)   # "very high traffic" threshold for caveats

def confidence(r):
    if r["problem_score"] >= 3 and r["impressions_90d"] >= 500:
        return "high"
    if r["problem_score"] >= 2:
        return "medium"
    return "low"

def would_be_wrong_if(r):
    notes = []
    if r["reason_codes"] == "low_ctr_visible_page|page_one_decay_risk" and r["avg_position"] <= 5:
        notes.append("low CTR may be normal at a top position (snippets/brand)")
    if r["impressions_90d"] >= p99_imp:
        notes.append("near-top traffic — priority partly from volume, not a fault")
    if r["avg_position"] == 0:
        notes.append("no position data")
    if r["word_count"] == 0:
        notes.append("length unknown")
    if r["problem_score"] < 2:
        notes.append("only one weak signal")
    return "; ".join(notes) if notes else "signals agree"

top20["confidence"]        = top20.apply(confidence, axis=1)
top20["would_be_wrong_if"] = top20.apply(would_be_wrong_if, axis=1)

review = top20[["baseline_rank", "content_id", "suggested_action", "reason_codes",
                "confidence", "would_be_wrong_if", "impressions_90d", "avg_position",
                "ctr", "days_since_last_update", "is_declining_label"]]
with pd.option_context("display.max_colwidth", 52, "display.width", 200):
    print(review.to_string(index=False))

print(f"\nTop-20 declining rate: {top20['is_declining_label'].mean():.3f}  vs base {base:.3f}")

 baseline_rank           content_id       suggested_action                                                reason_codes confidence                                                                                                     would_be_wrong_if  impressions_90d  avg_position  ctr  days_since_last_update  is_declining_label
             1 content_e3ff1b093148 refresh_and_review_ctr stale_visible_page|low_ctr_visible_page|page_one_decay_risk       high                                                                                                         signals agree             1408           7.8 0.28                     183                   1
             2 content_7f116ae1f6f5 refresh_and_review_ctr stale_visible_page|low_ctr_visible_page|page_one_decay_risk       high                                                                                                         signals agree              954           9.0 0.42                     301                   1
             3 c

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Takeaway:** the weak picks are the highest-traffic page-one pages, where a low CTR can be
perfectly normal (snippets / brand SERPs) — their priority comes partly from raw volume, not a
fault. The leakage assertion proves no `trend_*`, 30-day-window, or product columns entered the
score; `is_declining_label` is used for **evaluation only**.

In [4]:
# --- Weak picks: where the top-20 stands on thin ice (a good baseline review always finds some) ---
weak = top20[(top20["would_be_wrong_if"] != "signals agree")]
print(f"weak picks in top-20: {len(weak)} of 20")
print(weak[["baseline_rank", "content_id", "reason_codes", "problem_score",
            "impressions_90d", "avg_position", "would_be_wrong_if"]].to_string(index=False))

# --- Leakage check: the score reads ONLY pre-decision signals, no label/future/product columns ---
score_input_columns = ["impressions_90d", "days_since_last_update", "avg_position",
                       "ctr", "word_count", "content_age_days"]
forbidden = {"trend_direction", "trend_pct", "is_declining_label",
             "impressions_last_30d", "impressions_prev_30d",
             "clicks_last_30d", "clicks_prev_30d",
             "sessions_last_30d", "sessions_prev_30d",
             "provider_used", "model_used"}
leaked = set(score_input_columns) & forbidden
assert not leaked, f"LEAKAGE: score reads forbidden columns {leaked}"
print("\nleakage check PASSED")
print("  score reads only :", score_input_columns)
print("  kept out of score:", sorted(forbidden))

# --- The floor below the floor: a majority-class dummy ---
from sklearn.dummy import DummyClassifier
Xdummy = df[["impressions_90d"]].fillna(0)
dummy = DummyClassifier(strategy="most_frequent").fit(Xdummy, y)
print(f"\ndummy (predict majority) accuracy: {(dummy.predict(Xdummy) == y).mean():.3f}")

# --- Honest reading of the numbers ---
p20  = precision_at_k(df["baseline_action_score"], y, 20)
p50  = precision_at_k(df["baseline_action_score"], y, 50)
p100 = precision_at_k(df["baseline_action_score"], y, 100)
print(
    f"\nRead honestly: the very TOP of the queue is genuinely decline-enriched — "
    f"Precision@20 = {p20:.2f} vs a {base:.2f} base rate — because 'stale + still-visible' pages\n"
    f"(old, un-updated, yet still earning impressions) really do tend to be losing ground, and\n"
    f"days_since_last_update is a leakage-clean signal. But the lift FADES fast "
    f"(P@50 = {p50:.2f} ~ base, P@100 = {p100:.2f} < base):\n"
    f"deeper down, the common low-CTR / page-one-decay combo dominates and isn't decline-specific.\n"
    f"We deliberately exclude the last-30d/prev-30d windows that DEFINE 'declining', so this rule\n"
    f"ranks refresh PRIORITY, not decline — the honest, beatable bar the ML-08 model must clear\n"
    f"with signals that generalize across clients."
)

weak picks in top-20: 5 of 20
 baseline_rank           content_id                             reason_codes  problem_score  impressions_90d  avg_position                                                                                                     would_be_wrong_if
            16 content_5fe46e04994d low_ctr_visible_page|page_one_decay_risk              3           517715           4.2 low CTR may be normal at a top position (snippets/brand); near-top traffic — priority partly from volume, not a fault
            17 content_aaef01a50def low_ctr_visible_page|page_one_decay_risk              3           517109           5.4                                                           near-top traffic — priority partly from volume, not a fault
            18 content_8c19996aa890 low_ctr_visible_page|page_one_decay_risk              3           509252           2.5 low CTR may be normal at a top position (snippets/brand); near-top traffic — priority partly from volume, not a fault
      


dummy (predict majority) accuracy: 0.542

Read honestly: the very TOP of the queue is genuinely decline-enriched — Precision@20 = 0.90 vs a 0.54 base rate — because 'stale + still-visible' pages
(old, un-updated, yet still earning impressions) really do tend to be losing ground, and
days_since_last_update is a leakage-clean signal. But the lift FADES fast (P@50 = 0.54 ~ base, P@100 = 0.47 < base):
deeper down, the common low-CTR / page-one-decay combo dominates and isn't decline-specific.
We deliberately exclude the last-30d/prev-30d windows that DEFINE 'declining', so this rule
ranks refresh PRIORITY, not decline — the honest, beatable bar the ML-08 model must clear
with signals that generalize across clients.


## Self-check

- [x] Every section above is filled — the plain-words rule, the reason codes, and the code behind them.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) and rewrites
      `work/outputs/baseline_action_score.csv` each run.
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `content_id` /
      `client_id`, used for identification, never as score inputs.
- [x] Claims use care: `precision@K` is printed **next to the base rate**, the score uses **no
      fitted weights**, and a leakage assertion proves `trend_*`, the 30-day windows, and product
      fields never enter the score. The label is used for evaluation only.
- [x] The output CSV is a **build artifact**: gitignored by `work/**/*.csv` and blocked by the CI
      leak-guard, so it never enters git — the notebook regenerates it on demand.